# Benchmark pricing errors and significance

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


# **Load stock returns**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install pandas pyarrow scikit-learn joblib tqdm tables

import pandas as pd
import numpy as np
from sklearn.preprocessing import quantile_transform, StandardScaler
from tqdm import tqdm
from joblib import Parallel, delayed
from pathlib import Path

# Define the file path
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# Load the Parquet file into a DataFrame
df = pd.read_parquet(file_path, engine='pyarrow')

# Check the loaded data
print(df.head())

# Rename the 'month' column to 'date'
df = df.rename(columns={'month': 'date'})

# Ensure 'df' has a MultiIndex ['date', 'permno']
df = df.set_index(['date', 'permno'])

# Sort the MultiIndex
df = df.sort_index()

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]

# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Filter for dates starting from '1993'
df = df.loc[pd.IndexSlice['1975':, :], :]

# Create forward returns
df['fwd_ret_excess'] = df['ret_excess'].unstack('permno').shift(-1).stack(dropna=False)

# Define the function to apply quantile transformation for a single date
def process_date(date, df, characteristics):
    group = df.loc[date, characteristics]
    transformed = quantile_transform(group, copy=True, n_quantiles=group.shape[0])
    transformed_df = pd.DataFrame(transformed, columns=characteristics, index=group.index)
    transformed_df = transformed_df.mul(2).sub(1)
    transformed_df['date'] = date  # Add the date back to the DataFrame
    return transformed_df

# Initialize the progress bar
dates = df.index.get_level_values('date').unique()
pbar = tqdm(dates, desc="Processing Dates")

# Apply the transformation with parallel processing
normalized_data = Parallel(n_jobs=-1)(delayed(process_date)(date, df, characteristics) for date in pbar)

# Concatenate all the normalized data back into a single DataFrame
normalized_df = pd.concat(normalized_data)

# Reset and set the index to match the original DataFrame's multi-index
normalized_df = normalized_df.reset_index().set_index(['date', 'permno'])

# Ensure the index order matches the original DataFrame
normalized_df = normalized_df.reindex(df.index)

# Assign the normalized values back to the original DataFrame
df.loc[:, characteristics] = normalized_df[characteristics].values

# Close the progress bar
pbar.close()

# # Scale the variables
# scaler = StandardScaler()
# df[characteristics] = scaler.fit_transform(df[characteristics])

# Clip the returns columns
df.loc[:, ['ret_excess', 'fwd_ret_excess']] = df.loc[:, ['ret_excess', 'fwd_ret_excess']].clip(lower=-1, upper=1.0)

# Print the first few rows of the scaled DataFrame to verify
print(df.head())

# Add 'ret_excess' to the list of columns
columns_to_select = characteristics

# Select these columns from the DataFrame
characteristics_data = df[columns_to_select]

# Define a function to fill missing values with the median for each group
def fill_missing_with_median(group):
    return group.fillna(group.median())

# Apply the function to each group (each date) and preserve the index
df_filled = characteristics_data.groupby(level='date').apply(fill_missing_with_median)

# Remove the additional 'date' level in the index introduced by groupby
df_filled.index = df_filled.index.droplevel(0)

# Add the filled columns back to the original DataFrame
df.update(df_filled)

print(df)

dad = df

# Display the updated DataFrame to check the results
print(dad.head())


# **Loading the predictions from the IPCA and PCA models**

In [ ]:
from sys import pycache_prefix
############################################################
## LOADING AND COMBINING IPCA DATAFRAMES FOR EACH FACTOR ###
############################################################


import pandas as pd
from pathlib import Path
import numpy as np
from tqdm import tqdm

# Path to the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_IPCA')
hdf5_path = results_path / 'fwd_predictions.h5'

# Initialize dictionary to store combined dataframes for each factor
ipca_combined_dataframes = {}

# Load the dataframes for each factor and stack them together
for n_factors in tqdm(range(1, 7), desc='IPCA'):  # Adjust the range as needed
    combined_df = pd.DataFrame()  # Initialize an empty dataframe for each factor
    for fold in range(20):  # Assuming 20 folds
        key = f'fwd_predictions/n_factors_{n_factors}_fold_{fold}'
        try:
            with pd.HDFStore(hdf5_path, mode='r') as store:
                df = store.get(key)
                combined_df = pd.concat([combined_df, df], ignore_index=True)
        except KeyError:
            print(f"Data for {key} not found.")

    # Store the combined dataframe for the current factor in the dictionary
    ipca_combined_dataframes[n_factors] = combined_df

    # Save the combined dataframe for each factor if needed
    combined_hdf5_path = results_path / f'combined_predictions_n_factors_{n_factors}.h5'
    with pd.HDFStore(combined_hdf5_path, mode='w', complevel=9, complib='blosc:zstd') as store:
        store.put(f'combined_predictions/n_factors_{n_factors}', combined_df, format='table')

# Now `combined_dataframes` dictionary contains combined dataframes for each factor

##########################################################
## LOADING AND COMBINING PCA DATAFRAMES FOR EACH FACTOR ##
##########################################################

import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Path to the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_PCA')
hdf5_path = results_path / 'fwd_predictions.h5'

# Initialize dictionary to store combined dataframes for each factor
pca_combined_dataframes = {}

# Load the dataframes for each factor and stack them together
factors = [1, 2, 3, 4, 5, 6]  # List of factors
for n_factors in tqdm(factors, desc='PCA'):
    combined_df = pd.DataFrame()  # Initialize an empty dataframe for each factor
    for fold in range(20):  # Assuming 20 folds
        key = f'predictions/n_factors_{n_factors}_fold_{fold}'
        try:
            with pd.HDFStore(hdf5_path, mode='r') as store:
                df = store.get(key)
                combined_df = pd.concat([combined_df, df], ignore_index=True)
        except KeyError:
            print(f"Data for {key} not found.")

    # Store the combined dataframe for the current factor in the dictionary
    pca_combined_dataframes[n_factors] = combined_df

    # Save the combined dataframe for each factor if needed
    combined_hdf5_path = results_path / f'combined_predictions_n_factors_{n_factors}.h5'
    with pd.HDFStore(combined_hdf5_path, mode='w', complevel=9, complib='blosc:zstd') as store:
        store.put(f'combined_predictions/n_factors_{n_factors}', combined_df, format='table')

# Now `combined_dataframes` dictionary contains combined dataframes for each factor

###############################################################################################
## RENAMING COLLUMNS IN IPCA DF TO y_true and y_pred so they align with the rest of the code ##
###############################################################################################

factor = [1, 2, 3, 4, 5, 6]

for i in factor:
    drassle = ipca_combined_dataframes[i]
    drassle.rename(columns={'actual returns': 'y_true', 'predicted returns': 'y_pred'}, inplace=True)

print(ipca_combined_dataframes)





# **Loading the predictions from the PLS models**

In [ ]:
!pip install tables

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')


# Path to the saved HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/predictive_results_PLS')
hdf5_path = results_path / 'predictions.h5'

# Function to load and rename predictions from HDF5 for a specific factor
def load_and_process_predictions(hdf5_path, factor):
    try:
        with pd.HDFStore(hdf5_path, mode='r') as store:
            key = f'/pls_predictions/factors_{factor}'
            if key in store:
                predictions = store[key]
                predictions.rename(columns={'actual': 'y_true', 'predicted': 'y_pred'}, inplace=True)
                return predictions
            else:
                raise ValueError(f"No predictions found for {factor} factors in the HDF5 file.")
    except Exception as e:
        print(f"Error loading predictions: {e}")
        return None

# Initialize a dictionary to store combined dataframes for each factor
pls_combined_dataframes = {}

# Load the dataframes for each factor and store them in the dictionary
factors = [1, 2, 3, 4, 5, 6]  # List of factors
for n_factors in tqdm(factors, desc='PLS'):
    combined_df = load_and_process_predictions(hdf5_path, n_factors)
    if combined_df is not None:
        pls_combined_dataframes[n_factors] = combined_df

# Example: Accessing and displaying the first few rows of the predictions for a specific factor
factor_to_display = 4  # Specify the number of factors you want to display predictions for
if factor_to_display in pls_combined_dataframes:
    print(f"Displaying predictions for {factor_to_display} factors:")
    print(pls_combined_dataframes[factor_to_display].head())
else:
    print(f"No predictions available for {factor_to_display} factors.")

# Now `pls_combined_dataframes` dictionary contains processed dataframes for each factor,
# with columns 'y_true' and 'y_pred' for actual and predicted values, respectively.


In [ ]:
print(hdf5_path)


# **Loading Predictions for FF models**

In [ ]:
import pandas as pd
from pathlib import Path

# Path to the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_FF')
hdf5_path = results_path / 'fwd_predictions.h5'  # Adjusted to match the correct file name used during saving

# Factors sequences to use (same as used during saving)
factor_sequences = [
    ['Mkt-RF'],
    ['Mkt-RF', 'SMB'],
    ['Mkt-RF', 'SMB', 'HML'],
    ['Mkt-RF', 'SMB', 'HML', 'Mom'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA', 'RMW'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA', 'RMW','Mom']
]

# Dictionary to hold the concatenated predictions
concatenated_predictions = {}

# Load the predictions from the HDF5 file
with pd.HDFStore(hdf5_path, mode='r') as store:
    # List all keys in the store
    all_keys = store.keys()
    print(f"All keys in the store: {all_keys}")

    for n_factors in range(1, len(factor_sequences) + 1):
        factor_dfs = []
        for fold_key in all_keys:
            if f'/predictions/n_factors_{n_factors}_fold_' in fold_key:
                print(f"Loading key: {fold_key}")
                factor_df = store[fold_key]
                factor_dfs.append(factor_df)

        if factor_dfs:
            # Concatenate all folds for the current factor sequence
            concatenated_predictions[n_factors] = pd.concat(factor_dfs, ignore_index=True)
            print(f"Concatenated {len(factor_dfs)} dataframes for {n_factors} factors.")
        else:
            print(f"No dataframes found for n_factors_{n_factors}")

# Example: Access the concatenated DataFrame for the sequence of 6 factors
if 6 in concatenated_predictions:
    df_six_factors = concatenated_predictions[6]
    print(f"Head of concatenated DataFrame for 6 factors:\n{df_six_factors.head()}")
else:
    print("No concatenated predictions found for 6 factors.")


In [ ]:
concatenated_predictions[6]

In [ ]:
example_key

In [ ]:
# import pandas as pd

# ###############
# ## MERGE DFS ##
# ###############


# dad = df
# # Assuming dad dataframe and ipca_combined_dataframes are already defined

# # Function to check the date range and merge dataframes
# def check_date_range_and_merge(ipca_df, dad_df):
#     date_range = ipca_df.index.levels[0]
#     min_date = date_range.min()
#     max_date = date_range.max()
#     print(f"Date range of ipca_combined_dataframes: {min_date} to {max_date}")

#     # Ensure indices match
#     dad_df.index = dad_df.index.set_levels([pd.to_datetime(dad_df.index.levels[0]), dad_df.index.levels[1].astype(str)])
#     ipca_df.index = ipca_df.index.set_levels([pd.to_datetime(ipca_df.index.levels[0]), ipca_df.index.levels[1].astype(str)])

#     # Filter the dad dataframe to match the date range of ipca_df
#     filtered_dad = dad_df.loc[(dad_df.index.get_level_values('date') >= min_date) & (dad_df.index.get_level_values('date') <= max_date)]

#     # Check for common permno values
#     common_permnos = set(filtered_dad.index.get_level_values('permno')).intersection(set(ipca_df.index.get_level_values('permno')))
#     print(f"Number of common permno values: {len(common_permnos)}")

#     # Perform the merge if there are common permno values
#     if common_permnos:
#         merged_df = pd.merge(ipca_df, filtered_dad, left_index=True, right_index=True, how='inner')
#         return merged_df
#     else:
#         print("No common permno values found for the overlapping date range.")
#         return pd.DataFrame()  # return an empty dataframe if no common permno values are found

# # Apply the function to all ipca_combined_dataframes from 1 to 6
# merged_dataframes = {}
# for i in range(1, 7):
#     print(f"\nProcessing ipca_combined_dataframes[{i}]")
#     if pca_combined_dataframes[i] is not None:
#         merged_dataframes[i] = check_date_range_and_merge(pca_combined_dataframes[i], dad)
#         print(merged_dataframes[i].head())
#     else:
#         print(f"ipca_combined_dataframes[{i}] is None.")


In [ ]:
# import pandas as pd


# import numpy as np
# import pandas as pd
# from scipy import stats
# import matplotlib.pyplot as plt
# import os
# from pathlib import Path

# # Define the list of characteristic codes
# annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
#                 "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
#                 "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
#                 "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
#                 "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
#                 "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
#                 "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
# quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
#                    "stdacc", "stdcf"]
# monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
#                  "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
#                  "zerotrade"]

# # Combine all codes
# all_codes = annual_codes + quarterly_codes + monthly_codes
# characteristics = [f'characteristic_{code}' for code in all_codes]

# # Calculate x_t for a given month using Ridge Regression
# def calculate_xt_y(df, date, target, alpha=0):
#     df_month = df.loc[date]
#     r_t = df_month[target].values
#     characteristic_columns = df_month.drop(columns=[target]).columns
#     Z_t_minus_1 = df_month.drop(columns=[target]).values
#     Z_t_minus_1_T = Z_t_minus_1.T
#     Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
#     Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])
#     Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)
#     x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t
#     return x_t, characteristic_columns

# # Calculate x_t for all unique dates in the dataset
# def calculate_xt_all_dates(df, target, alpha=1e-6):
#     xt_results = {}
#     characteristic_columns = None
#     unique_dates = df.index.get_level_values('date').unique()
#     for date in unique_dates:
#         x_t, characteristic_columns = calculate_xt_y(df, date, target, alpha)
#         xt_results[date] = x_t
#     xt_df = pd.DataFrame.from_dict(xt_results, orient='index')
#     if characteristic_columns is not None:
#         xt_df.columns = characteristic_columns
#     xt_df.index.name = 'date'
#     return xt_df

# # Function to calculate R^2
# def calculate_r2(xt_true, xt_pred):
#     y_true = xt_true.values.flatten()
#     y_pred = xt_pred.values.flatten()
#     residuals = y_true - y_pred
#     ssr = np.sum(np.square(residuals))
#     tss = np.sum(np.square(y_true - np.mean(y_true)))
#     r_squared = 1 - (ssr / tss)
#     return r_squared

# # Function to merge dataframes
# def merge_dataframes(ipca_combined_dataframes, dad):
#     merged_dataframes = {}
#     for i in range(1, len(ipca_combined_dataframes) + 1):
#         print(f"\nProcessing ipca_combined_dataframes[{i}]")
#         if ipca_combined_dataframes[i] is not None:
#             date_range = ipca_combined_dataframes[i].index.get_level_values('date')
#             min_date = date_range.min()
#             max_date = date_range.max()
#             print(f"Date range of ipca_combined_dataframes[{i}]: {min_date} to {max_date}")

#             # Ensure indices match
#             dad.index = dad.index.set_levels([pd.to_datetime(dad.index.levels[0]), dad.index.levels[1].astype(str)])
#             ipca_combined_dataframes[i].index = ipca_combined_dataframes[i].index.set_levels([pd.to_datetime(ipca_combined_dataframes[i].index.levels[0]), ipca_combined_dataframes[i].index.levels[1].astype(str)])

#             # Filter the dad dataframe to match the date range of ipca_combined_dataframes[i]
#             filtered_dad = dad.loc[(dad.index.get_level_values('date') >= min_date) & (dad.index.get_level_values('date') <= max_date)]

#             # Check for common permno values
#             common_permnos = set(filtered_dad.index.get_level_values('permno')).intersection(set(ipca_combined_dataframes[i].index.get_level_values('permno')))
#             print(f"Number of common permno values: {len(common_permnos)}")

#             # Perform the merge if there are common permno values
#             if common_permnos:
#                 merged_df = pd.merge(ipca_combined_dataframes[i], filtered_dad, left_index=True, right_index=True, how='inner')
#                 merged_dataframes[i] = merged_df
#                 print(merged_df.head())
#             else:
#                 print(f"No common permno values found for the overlapping date range in ipca_combined_dataframes[{i}].")
#                 merged_dataframes[i] = pd.DataFrame()  # return an empty dataframe if no common permno values are found
#         else:
#             print(f"ipca_combined_dataframes[{i}] is None.")
#             merged_dataframes[i] = pd.DataFrame()
#     return merged_dataframes


# # Process merged dataframes and calculate statistics
# def process_all_merged_dataframes(merged_dataframes):
#     results = []
#     for i in range(1, len(merged_dataframes) + 1):
#         if merged_dataframes[i] is not None and not merged_dataframes[i].empty:
#             characteristics_data_pred = merged_dataframes[i][characteristics + ["y_pred"]]
#             characteristics_data_true = merged_dataframes[i][characteristics + ["y_true"]]
#             xt_true = calculate_xt_all_dates(characteristics_data_true, 'y_true')
#             xt_pred = calculate_xt_all_dates(characteristics_data_pred, 'y_pred')
#             xt_r2 = calculate_r2(xt_true, xt_pred)
#             expected_xt_true = xt_true.mean()
#             expected_xt_pred = xt_pred.mean()
#             alpha = expected_xt_true - expected_xt_pred
#             epsilon = xt_true - expected_xt_pred
#             alpha_epsilon = epsilon.mean()
#             s_epsilon = epsilon.std(ddof=1)
#             n = len(epsilon)
#             SE_alpha = s_epsilon / np.sqrt(n)
#             t_alpha = alpha_epsilon / SE_alpha
#             alpha_level = 0.05
#             degrees_of_freedom = n - 1
#             t_critical = stats.t.ppf(1 - alpha_level / 2, degrees_of_freedom)
#             significant_mask = np.abs(t_alpha) > t_critical
#             results.append((expected_xt_true.values, alpha.values, significant_mask, f'factor_{i}'))
#         else:
#             print(f"merged_dataframe[{i}] is None or empty.")
#     return results

# # Function to plot results
# def plot_results(expected_xt_true, alpha, significant_mask, factor_name):
#     plt.figure(figsize=(10, 6))
#     colors = np.where(significant_mask, 'red', 'blue')
#     plt.scatter(expected_xt_true, alpha, c=colors, alpha=0.7)
#     p = np.polyfit(expected_xt_true, alpha, 1)
#     plt.plot(expected_xt_true, np.polyval(p, expected_xt_true), color='green')
#     red_patch = plt.Line2D([0], [0], marker='o', color='w', label=f'Significant Alphas: {significant_mask.sum()}',
#                            markerfacecolor='red', markersize=10)
#     blue_patch = plt.Line2D([0], [0], marker='o', color='w', label=f'Non-Significant Alphas: {len(significant_mask) - significant_mask.sum()}',
#                             markerfacecolor='blue', markersize=10)
#     line_patch = plt.Line2D([0], [0], color='green', label='Fitted Line')
#     plt.legend(handles=[red_patch, blue_patch])
#     plt.xlabel('Realized returns of $x_t$')
#     plt.ylabel('Alpha')
#     plt.title(f'Scatter Plot of Alphas vs. xt_true for {factor_name}')
#     plt.grid(True)
#     results_path = Path("./results")
#     os.makedirs(results_path, exist_ok=True)
#     plt.savefig(results_path / f'Scatter_plot_for_{factor_name}.png')
#     plt.show()

# # Main function to process all factors and plot results
# def process_ipca_and_dad(ipca_combined_dataframes, dad):
#     ipca_combined_dataframes = set_indices(ipca_combined_dataframes)
#     merged_dataframes = merge_dataframes(ipca_combined_dataframes, dad)
#     results = process_all_merged_dataframes(merged_dataframes)
#     for expected_xt_true, alpha, significant_mask, factor_name in results:
#         plot_results(expected_xt_true, alpha, significant_mask, factor_name)

# # Assuming the necessary functions like `set_indices` and `merge_dataframes` are defined and used correctly

# # Function to set the indices if they are not already set
# def set_indices(ipca_combined_dataframes):
#     for i in range(1, len(ipca_combined_dataframes) + 1):
#         if ipca_combined_dataframes[i] is not None:
#             if not isinstance(ipca_combined_dataframes[i].index, pd.MultiIndex) or \
#                'date' not in ipca_combined_dataframes[i].index.names or \
#                'permno' not in ipca_combined_dataframes[i].index.names:
#                 print(f"Setting indices for ipca_combined_dataframes[{i}]")
#                 ipca_combined_dataframes[i] = ipca_combined_dataframes[i].set_index(['date', 'permno'])
#             else:
#                 print(f"indices for ipca_combined_dataframes[{i}] are already set.")
#     return ipca_combined_dataframes



# # Call the function to set indices
# ipca_combined_dataframes = set_indices(ipca_combined_dataframes)


# # Call the main function to process the data
# process_ipca_and_dad(ipca_combined_dataframes, dad)





**look in the results folder in google colab to see if the plots are saved**

In [ ]:
from google.colab import files
!ls ./results


**Download the plots saved into the results folder in google colab**

In [ ]:
# If you want to download all files in the directory, you can zip them first
!zip -r results.zip ./results

# Download the zip file
files.download('results.zip')

**One plot i got right at first**

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]

# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Calculate x_t for a given month using Ridge Regression
def calculate_xt_y(df, date, target, alpha=0):
    # Filter DataFrame for the given date
    df_month = df.loc[date]

    # Separate returns and characteristics
    r_t = df_month[target].values
    characteristic_columns = df_month.drop(columns=[target]).columns
    Z_t_minus_1 = df_month.drop(columns=[target]).values

    # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)

    # Calculate x_t
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t

    return x_t, characteristic_columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates(df, target, alpha=1e-6):
    xt_results = {}
    characteristic_columns = None

    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt_y(df, date, target, alpha)
        xt_results[date] = x_t

    # Convert dictionary to DataFrame
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')

    # Assign original characteristic names as column names
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns

    # Preserve the index name
    xt_df.index.name = 'date'

    return xt_df

# Function to calculate R^2
def calculate_r2(xt_true, xt_pred):
    y_true = xt_true.values.flatten()
    y_pred = xt_pred.values.flatten()
    residuals = y_true - y_pred
    ssr = np.sum(np.square(residuals))
    tss = np.sum(np.square(y_true - np.mean(y_true)))
    r_squared = 1 - (ssr / tss)
    return r_squared

# Process a single merged dataframe for demonstration
def process_single_merged_dataframe(merged_df, i=1):
    if merged_df is not None and not merged_df.empty:
        characteristics_data_pred = merged_df[characteristics + ["y_pred"]]
        characteristics_data_true = merged_df[characteristics + ["y_true"]]

        xt_true = calculate_xt_all_dates(characteristics_data_true, 'y_true')
        xt_pred = calculate_xt_all_dates(characteristics_data_pred, 'y_pred')
        xt_r2 = calculate_r2(xt_true, xt_pred)
        print(f'$R^2$ for $x_t$ for merged_dataframe[{i}]: {xt_r2}')

        expected_xt_true = xt_true.mean()
        expected_xt_pred = xt_pred.mean()
        alpha = expected_xt_true - expected_xt_pred

        epsilon = xt_true - expected_xt_pred
        alpha_epsilon = epsilon.mean()
        s_epsilon = epsilon.std(ddof=1)
        n = len(epsilon)
        SE_alpha = s_epsilon / np.sqrt(n)
        t_alpha = alpha_epsilon / SE_alpha

        alpha_level = 0.05
        degrees_of_freedom = n - 1
        t_critical = stats.t.ppf(1 - alpha_level / 2, degrees_of_freedom)
        significant_mask = np.abs(t_alpha) > t_critical

        return expected_xt_true.values, alpha.values, significant_mask

    else:
        print(f"merged_dataframe[{i}] is None or empty.")
        return None, None, None

# Function to plot results
def plot_results(expected_xt_true, alpha, significant_mask, factor_name):
    plt.figure(figsize=(10, 6))
    colors = np.where(significant_mask, 'red', 'blue')
    plt.scatter(expected_xt_true, alpha, c=colors, alpha=0.7)

    # Fitting a line to the data
    p = np.polyfit(expected_xt_true, alpha, 1)
    plt.plot(expected_xt_true, np.polyval(p, expected_xt_true), color='green')

    # Create legend with the number of significant alphas
    red_patch = plt.Line2D([0], [0], marker='o', color='w', label=f'Significant Alphas: {significant_mask.sum()}',
                           markerfacecolor='red', markersize=10)
    blue_patch = plt.Line2D([0], [0], marker='o', color='w', label=f'Non-Significant Alphas: {len(significant_mask) - significant_mask.sum()}',
                            markerfacecolor='blue', markersize=10)
    line_patch = plt.Line2D([0], [0], color='green', label='Fitted Line')
    plt.legend(handles=[red_patch, blue_patch, line_patch])

    plt.xlabel('xt_true')
    plt.ylabel('Alpha')
    plt.title(f'Scatter Plot of Alphas vs. xt_true for {factor_name}')
    plt.grid(True)

    # Create the results path if it doesn't exist
    results_path = Path("./results")
    os.makedirs(results_path, exist_ok=True)
    plt.savefig(results_path / f'Scatter_plot_for_{factor_name}.png')
    plt.show()

# Example usage:
# Assuming merged_dataframes[1] is already available
expected_xt_true, alpha, significant_mask = process_single_merged_dataframe(merged_dataframes[1], i=1)
if expected_xt_true is not None and alpha is not None and significant_mask is not None:
    plot_results(expected_xt_true, alpha, significant_mask, 'factor_1')


**First code with all plots**

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import os
from pathlib import Path

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]

# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Calculate x_t for a given month using Ridge Regression
def calculate_xt_y(df, date, target, alpha=0):
    df_month = df.loc[date]
    r_t = df_month[target].values
    characteristic_columns = df_month.drop(columns=[target]).columns
    Z_t_minus_1 = df_month.drop(columns=[target]).values
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])
    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t
    return x_t, characteristic_columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates(df, target, alpha=1e-6):
    xt_results = {}
    characteristic_columns = None
    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt_y(df, date, target, alpha)
        xt_results[date] = x_t
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns
    xt_df.index.name = 'date'
    return xt_df

# Function to calculate R^2
def calculate_r2(xt_true, xt_pred):
    y_true = xt_true.values.flatten()
    y_pred = xt_pred.values.flatten()
    residuals = y_true - y_pred
    ssr = np.sum(np.square(residuals))
    tss = np.sum(np.square(y_true - np.mean(y_true)))
    r_squared = 1 - (ssr / tss)
    return r_squared

# Process merged dataframes and calculate statistics
def process_all_merged_dataframes(merged_dataframes):
    results = []
    for i in range(1, len(merged_dataframes) + 1):
        if merged_dataframes[i] is not None and not merged_dataframes[i].empty:
            characteristics_data_pred = merged_dataframes[i][characteristics + ["y_pred"]]
            characteristics_data_true = merged_dataframes[i][characteristics + ["y_true"]]
            xt_true = calculate_xt_all_dates(characteristics_data_true, 'y_true')
            xt_pred = calculate_xt_all_dates(characteristics_data_pred, 'y_pred')
            xt_r2 = calculate_r2(xt_true, xt_pred)
            expected_xt_true = xt_true.mean()
            expected_xt_pred = xt_pred.mean()
            alpha = expected_xt_true - expected_xt_pred
            epsilon = xt_true - expected_xt_pred
            alpha_epsilon = epsilon.mean()
            s_epsilon = epsilon.std(ddof=1)
            n = len(epsilon)
            SE_alpha = s_epsilon / np.sqrt(n)
            t_alpha = alpha_epsilon / SE_alpha
            alpha_level = 0.05
            degrees_of_freedom = n - 1
            t_critical = stats.t.ppf(1 - alpha_level / 2, degrees_of_freedom)
            significant_mask = np.abs(t_alpha) > t_critical
            results.append((expected_xt_true.values, alpha.values, significant_mask, f'factor_{i}'))
        else:
            print(f"merged_dataframe[{i}] is None or empty.")
    return results

# Function to plot results
def plot_results(expected_xt_true, alpha, significant_mask, factor_name):
    plt.figure(figsize=(10, 6))
    colors = np.where(significant_mask, 'red', 'blue')
    plt.scatter(expected_xt_true, alpha, c=colors, alpha=0.7)
    p = np.polyfit(expected_xt_true, alpha, 1)
    plt.plot(expected_xt_true, np.polyval(p, expected_xt_true), color='green')
    red_patch = plt.Line2D([0], [0], marker='o', color='w', label=f'Significant Alphas: {significant_mask.sum()}',
                           markerfacecolor='red', markersize=10)
    blue_patch = plt.Line2D([0], [0], marker='o', color='w', label=f'Non-Significant Alphas: {len(significant_mask) - significant_mask.sum()}',
                            markerfacecolor='blue', markersize=10)
    line_patch = plt.Line2D([0], [0], color='green', label='Fitted Line')
    plt.legend(handles=[red_patch, blue_patch, line_patch])
    plt.xlabel('xt_true')
    plt.ylabel('Alpha')
    plt.title(f'Scatter Plot of Alphas vs. xt_true for {factor_name}')
    plt.grid(True)
    results_path = Path("./results")
    os.makedirs(results_path, exist_ok=True)
    plt.savefig(results_path / f'Scatter_plot_for_{factor_name}.png')
    plt.show()

# Main function to process all factors and plot results
def process_ipca_and_dad(ipca_combined_dataframes, dad):
    ipca_combined_dataframes = set_indices(ipca_combined_dataframes)
    merged_dataframes = merge_dataframes(ipca_combined_dataframes, dad)
    results = process_all_merged_dataframes(merged_dataframes)
    for expected_xt_true, alpha, significant_mask, factor_name in results:
        plot_results(expected_xt_true, alpha, significant_mask, factor_name)

# Assuming the necessary functions like `set_indices` and `merge_dataframes` are defined and used correctly

# Call the main function to process the data
process_ipca_and_dad(ipca_combined_dataframes, dad)


# **Second code with all plots and tables for alphas of $x_t$s**

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import os
from pathlib import Path

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]

# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Calculate x_t for a given month using Ridge Regression
def calculate_xt_y(df, date, target, alpha=0):
    df_month = df.loc[date]
    r_t = df_month[target].values
    characteristic_columns = df_month.drop(columns=[target]).columns
    Z_t_minus_1 = df_month.drop(columns=[target]).values
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])
    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t
    return x_t, characteristic_columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates(df, target, alpha=1e-6):
    xt_results = {}
    characteristic_columns = None
    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt_y(df, date, target, alpha)
        xt_results[date] = x_t
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns
    xt_df.index.name = 'date'
    return xt_df

# Function to calculate R^2
def calculate_r2(xt_true, xt_pred):
    y_true = xt_true.values.flatten()
    y_pred = xt_pred.values.flatten()
    residuals = y_true - y_pred
    ssr = np.sum(np.square(residuals))
    tss = np.sum(np.square(y_true - np.mean(y_true)))
    r_squared = 1 - (ssr / tss)
    return r_squared

# Process merged dataframes and calculate statistics
def process_all_merged_dataframes(merged_dataframes):
    r2_results = {}
    significant_alpha_counts = {}
    alphamale = {}
    avg_t_critical_alpha = {}
    prop_of_sign_ts = {}

    results = []
    for i in range(1, len(merged_dataframes) + 1):
        if merged_dataframes[i] is not None and not merged_dataframes[i].empty:
            characteristics_data_pred = merged_dataframes[i][characteristics + ["y_pred"]]
            characteristics_data_true = merged_dataframes[i][characteristics + ["y_true"]]
            xt_true = calculate_xt_all_dates(characteristics_data_true, 'y_true')
            xt_pred = calculate_xt_all_dates(characteristics_data_pred, 'y_pred')
            xt_r2 = calculate_r2(xt_true, xt_pred)

            expected_xt_true = xt_true.mean()
            expected_xt_pred = xt_pred.mean()
            alpha = expected_xt_true - expected_xt_pred

            epsilon = xt_true - expected_xt_pred
            alpha_epsilon = epsilon.mean()
            s_epsilon = epsilon.std(ddof=1)
            n = len(epsilon)
            SE_alpha = s_epsilon / np.sqrt(n)
            t_alpha = alpha_epsilon / SE_alpha
            alpha_level = 0.05
            degrees_of_freedom = n - 1
            t_critical = 3
            # t_critical = stats.t.ppf(1 - alpha_level / 2, degrees_of_freedom)
            significant_mask = np.abs(t_alpha) > t_critical

            r2_results[f'factor_{i}'] = xt_r2
            significant_alpha_counts[f'factor_{i}'] = significant_mask.sum()
            alphamale[f'factor_{i}'] = np.abs(alpha).mean()
            avg_t_critical_alpha[f'factor_{i}'] = np.abs(t_alpha).mean()
            prop_of_sign_ts[f'factor_{i}'] = significant_mask.mean()

            results.append((expected_xt_true.values, alpha.values, significant_mask, f'factor_{i}'))
        else:
            print(f"merged_dataframe[{i}] is None or empty.")

    return results, r2_results, significant_alpha_counts, alphamale, avg_t_critical_alpha, prop_of_sign_ts

# Function to plot results
def plot_results(expected_xt_true, alpha, significant_mask, factor_name):
    plt.figure(figsize=(10, 6))
    colors = np.where(significant_mask, 'red', 'blue')
    plt.scatter(expected_xt_true, alpha, c=colors, alpha=0.7)
    p = np.polyfit(expected_xt_true, alpha, 1)
    plt.plot(expected_xt_true, np.polyval(p, expected_xt_true), color='green')
    red_patch = plt.Line2D([0], [0], marker='o', color='w', label=f'Significant Alphas: {significant_mask.sum()}',
                           markerfacecolor='red', markersize=10)
    blue_patch = plt.Line2D([0], [0], marker='o', color='w', label=f'Non-Significant Alphas: {len(significant_mask) - significant_mask.sum()}',
                            markerfacecolor='blue', markersize=10)
    plt.legend(handles=[red_patch, blue_patch])
    plt.xlabel('Realized Mean Returns')
    plt.ylabel('Alpha')
    plt.title(f'Alphas vs. Realized Mean returns for FF{factor_name}')
    plt.grid(True)
    results_path = Path("./results")
    os.makedirs(results_path, exist_ok=True)
    plt.savefig(results_path / f'Scatter_plot_for_{factor_name}.png')
    plt.show()

# Function to generate LaTeX code for pivoted DataFrames
def generate_latex_code(df):
    return df.to_latex()

# Function to set the indices if they are not already set
def set_indices(ipca_combined_dataframes):
    for i in range(1, len(ipca_combined_dataframes) + 1):
        if ipca_combined_dataframes[i] is not None:
            if not isinstance(ipca_combined_dataframes[i].index, pd.MultiIndex) or \
               'date' not in ipca_combined_dataframes[i].index.names or \
               'permno' not in ipca_combined_dataframes[i].index.names:
                print(f"Setting indices for ipca_combined_dataframes[{i}]")
                ipca_combined_dataframes[i] = ipca_combined_dataframes[i].set_index(['date', 'permno'])
            else:
                print(f"indices for ipca_combined_dataframes[{i}] are already set.")
    return ipca_combined_dataframes

# Function to merge dataframes
def merge_dataframes(ipca_combined_dataframes, dad):
    merged_dataframes = {}
    for i in range(1, len(ipca_combined_dataframes) + 1):
        print(f"\nProcessing ipca_combined_dataframes[{i}]")
        if ipca_combined_dataframes[i] is not None:
            date_range = ipca_combined_dataframes[i].index.get_level_values('date')
            min_date = date_range.min()
            max_date = date_range.max()
            print(f"Date range of ipca_combined_dataframes[{i}]: {min_date} to {max_date}")

            # Ensure indices match
            dad.index = dad.index.set_levels([pd.to_datetime(dad.index.levels[0]), dad.index.levels[1].astype(str)])
            ipca_combined_dataframes[i].index = ipca_combined_dataframes[i].index.set_levels([pd.to_datetime(ipca_combined_dataframes[i].index.levels[0]), ipca_combined_dataframes[i].index.levels[1].astype(str)])

            # Filter the dad dataframe to match the date range of ipca_combined_dataframes[i]
            filtered_dad = dad.loc[(dad.index.get_level_values('date') >= min_date) & (dad.index.get_level_values('date') <= max_date)]

            # Check for common permno values
            common_permnos = set(filtered_dad.index.get_level_values('permno')).intersection(set(ipca_combined_dataframes[i].index.get_level_values('permno')))
            print(f"Number of common permno values: {len(common_permnos)}")

            # Perform the merge if there are common permno values
            if common_permnos:
                merged_df = pd.merge(ipca_combined_dataframes[i], filtered_dad, left_index=True, right_index=True, how='inner')
                merged_dataframes[i] = merged_df
                print(merged_df.head())
            else:
                print(f"No common permno values found for the overlapping date range in ipca_combined_dataframes[{i}].")
                merged_dataframes[i] = pd.DataFrame()  # return an empty dataframe if no common permno values are found
        else:
            print(f"ipca_combined_dataframes[{i}] is None.")
            merged_dataframes[i] = pd.DataFrame()
    return merged_dataframes


# Main function to process all factors and plot results
def process_ipca_and_dad(ipca_combined_dataframes, dad):
    ipca_combined_dataframes = set_indices(ipca_combined_dataframes)
    merged_dataframes = merge_dataframes(ipca_combined_dataframes, dad)
    results, r2_results, significant_alpha_counts, alphamale, avg_t_critical_alpha, prop_of_sign_ts = process_all_merged_dataframes(merged_dataframes)

    # Generate DataFrames for the metrics
    r2_df = pd.DataFrame.from_dict(r2_results, orient='index', columns=['R^2'])
    significant_alpha_counts_df = pd.DataFrame.from_dict(significant_alpha_counts, orient='index', columns=['Significant Alpha Counts'])
    alphamale_df = pd.DataFrame.from_dict(alphamale, orient='index', columns=['Alpha Male'])
    avg_t_critical_alpha_df = pd.DataFrame.from_dict(avg_t_critical_alpha, orient='index', columns=['Average T-Critical Alpha'])
    prop_of_sign_ts_df = pd.DataFrame.from_dict(prop_of_sign_ts, orient='index', columns=['Proportion of Significant T-Statistics'])

    # Pivot the DataFrames
    r2_df_pivoted = r2_df.T
    significant_alpha_counts_df_pivoted = significant_alpha_counts_df.T
    alphamale_df_pivoted = alphamale_df.T
    avg_t_critical_alpha_df_pivoted = avg_t_critical_alpha_df.T
    prop_of_sign_ts_df_pivoted = prop_of_sign_ts_df.T

    # Generate LaTeX code for pivoted DataFrames
    latex_r2 = generate_latex_code(r2_df_pivoted)
    latex_significant_alpha_counts = generate_latex_code(significant_alpha_counts_df_pivoted)
    latex_alphamale = generate_latex_code(alphamale_df_pivoted)
    latex_avg_t_critical_alpha = generate_latex_code(avg_t_critical_alpha_df_pivoted)
    latex_prop_of_sign_ts = generate_latex_code(prop_of_sign_ts_df_pivoted)

    # Print LaTeX code
    print("LaTeX code for R^2:")
    print(latex_r2)
    print("\nLaTeX code for Significant Alpha Counts:")
    print(latex_significant_alpha_counts)
    print("\nLaTeX code for Alpha Male:")
    print(latex_alphamale)
    print("\nLaTeX code for Average T-Critical Alpha:")
    print(latex_avg_t_critical_alpha)
    print("\nLaTeX code for Proportion of Significant T-Statistics:")
    print(latex_prop_of_sign_ts)

    # Plot results
    for expected_xt_true, alpha, significant_mask, factor_name in results:
        plot_results(expected_xt_true, alpha, significant_mask, factor_name)

# Assuming the necessary functions like `set_indices` and `merge_dataframes` are defined and used correctly

# Call the main function to process the data
process_ipca_and_dad(concatenated_predictions, dad)


**Calculating $R^2$ for individual stock returns**

In [ ]:
from sys import pycache_prefix
############################################################
## LOADING AND COMBINING IPCA DATAFRAMES FOR EACH FACTOR ###
############################################################


import pandas as pd
from pathlib import Path
import numpy as np
from tqdm import tqdm

# Path to the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_IPCA')
hdf5_path = results_path / 'fwd_predictions.h5'

# Initialize dictionary to store combined dataframes for each factor
ipca_combined_dataframes = {}

# Load the dataframes for each factor and stack them together
for n_factors in tqdm(range(1, 7), desc='IPCA'):  # Adjust the range as needed
    combined_df = pd.DataFrame()  # Initialize an empty dataframe for each factor
    for fold in range(20):  # Assuming 20 folds
        key = f'fwd_predictions/n_factors_{n_factors}_fold_{fold}'
        try:
            with pd.HDFStore(hdf5_path, mode='r') as store:
                df = store.get(key)
                combined_df = pd.concat([combined_df, df], ignore_index=True)
        except KeyError:
            print(f"Data for {key} not found.")

    # Store the combined dataframe for the current factor in the dictionary
    ipca_combined_dataframes[n_factors] = combined_df

    # Save the combined dataframe for each factor if needed
    combined_hdf5_path = results_path / f'combined_predictions_n_factors_{n_factors}.h5'
    with pd.HDFStore(combined_hdf5_path, mode='w', complevel=9, complib='blosc:zstd') as store:
        store.put(f'combined_predictions/n_factors_{n_factors}', combined_df, format='table')

# Now `combined_dataframes` dictionary contains combined dataframes for each factor

##########################################################
## LOADING AND COMBINING PCA DATAFRAMES FOR EACH FACTOR ##
##########################################################

import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Path to the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_PCA')
hdf5_path = results_path / 'fwd_predictions.h5'

# Initialize dictionary to store combined dataframes for each factor
pca_combined_dataframes = {}

# Load the dataframes for each factor and stack them together
factors = [1, 2, 3, 4, 5, 6]  # List of factors
for n_factors in tqdm(factors, desc='PCA'):
    combined_df = pd.DataFrame()  # Initialize an empty dataframe for each factor
    for fold in range(20):  # Assuming 20 folds
        key = f'predictions/n_factors_{n_factors}_fold_{fold}'
        try:
            with pd.HDFStore(hdf5_path, mode='r') as store:
                df = store.get(key)
                combined_df = pd.concat([combined_df, df], ignore_index=True)
        except KeyError:
            print(f"Data for {key} not found.")

    # Store the combined dataframe for the current factor in the dictionary
    pca_combined_dataframes[n_factors] = combined_df

    # Save the combined dataframe for each factor if needed
    combined_hdf5_path = results_path / f'combined_predictions_n_factors_{n_factors}.h5'
    with pd.HDFStore(combined_hdf5_path, mode='w', complevel=9, complib='blosc:zstd') as store:
        store.put(f'combined_predictions/n_factors_{n_factors}', combined_df, format='table')

# Now `combined_dataframes` dictionary contains combined dataframes for each factor

###############################################################################################
## RENAMING COLLUMNS IN IPCA DF TO y_true and y_pred so they align with the rest of the code ##
###############################################################################################

factor = [1, 2, 3, 4, 5, 6]

for i in factor:
    drassle = ipca_combined_dataframes[i]
    drassle.rename(columns={'actual returns': 'y_true', 'predicted returns': 'y_pred'}, inplace=True)

print(ipca_combined_dataframes)





#**Calculation of $R^2$ for IPCA and PCA models**

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from tqdm import tqdm

# Path to the HDF5 file
results_ipca_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_IPCA')
results_pca_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_PCA')
hdf5_ipca_path = results_ipca_path / 'fwd_predictions.h5'
hdf5_pca_path = results_pca_path / 'fwd_predictions.h5'

# Function to load and combine dataframes for each factor
def load_and_combine_dataframes(hdf5_path, key_template, num_factors, num_folds):
    combined_dataframes = {}
    for n_factors in tqdm(range(1, num_factors + 1), desc='Loading dataframes'):
        combined_df = pd.DataFrame()
        for fold in range(num_folds):
            key = key_template.format(n_factors=n_factors, fold=fold)
            try:
                with pd.HDFStore(hdf5_path, mode='r') as store:
                    df = store.get(key)
                    combined_df = pd.concat([combined_df, df], ignore_index=True)
            except KeyError:
                print(f"Data for {key} not found.")
        combined_dataframes[n_factors] = combined_df
    return combined_dataframes

# Load IPCA dataframes
ipca_combined_dataframes = load_and_combine_dataframes(hdf5_ipca_path, 'fwd_predictions/n_factors_{n_factors}_fold_{fold}', 6, 20)

# Load PCA dataframes
pca_combined_dataframes = load_and_combine_dataframes(hdf5_pca_path, 'predictions/n_factors_{n_factors}_fold_{fold}', 6, 20)

# Rename columns in IPCA dataframes to y_true and y_pred
for i in range(1, 7):
    ipca_combined_dataframes[i].rename(columns={'actual returns': 'y_true', 'predicted returns': 'y_pred'}, inplace=True)

# Function to calculate R^2 for each factor
def calculate_r2(df):
    y_true = df['y_true']
    y_pred = df['y_pred']
    ssr = np.sum((y_true - y_pred) ** 2)
    tss = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ssr / tss)

# Calculate R^2 for each factor in IPCA and PCA dataframes
r2_ipca_results = {f'IPCA_factor_{i}': calculate_r2(df) for i, df in ipca_combined_dataframes.items() if not df.empty}
r2_pca_results = {f'PCA_factor_{i}': calculate_r2(df) for i, df in pca_combined_dataframes.items() if not df.empty}

# Create DataFrames for the results
r2_ipca_df = pd.DataFrame.from_dict(r2_ipca_results, orient='index', columns=['R^2'])
r2_pca_df = pd.DataFrame.from_dict(r2_pca_results, orient='index', columns=['R^2'])

# Pivot the dataframes
r2_ipca_df_pivoted = r2_ipca_df.T
r2_pca_df_pivoted = r2_pca_df.T

# Function to convert DataFrame to LaTeX code
def dataframe_to_latex(df):
    return df.to_latex(index=False)

# Generate LaTeX code for the pivoted dataframes
r2_ipca_latex = dataframe_to_latex(r2_ipca_df_pivoted)
r2_pca_latex = dataframe_to_latex(r2_pca_df_pivoted)

# Display the results
print("R^2 IPCA DataFrame:")
print(r2_ipca_df_pivoted)

print("\nR^2 PCA DataFrame:")
print(r2_pca_df_pivoted)

print("\nR^2 IPCA LaTeX:")
print(r2_ipca_latex)

print("\nR^2 PCA LaTeX:")
print(r2_pca_latex)


#**Calculation of $R^2$ of PLS**

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
!pip install tables
from google.colab import drive
drive.mount('/content/drive')

# Path to the saved HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/predictive_results_PLS')
hdf5_path = results_path / 'predictions.h5'

# Function to load and rename predictions from HDF5 for a specific factor
def load_and_process_predictions(hdf5_path, factor):
    try:
        with pd.HDFStore(hdf5_path, mode='r') as store:
            key = f'/pls_predictions/factors_{factor}'
            if key in store:
                predictions = store[key]
                predictions.rename(columns={'actual': 'y_true', 'predicted': 'y_pred'}, inplace=True)
                return predictions
            else:
                raise ValueError(f"No predictions found for {factor} factors in the HDF5 file.")
    except Exception as e:
        print(f"Error loading predictions: {e}")
        return None

# Function to calculate R² for each factor
def calculate_r2(df):
    y_true = df['y_true']
    y_pred = df['y_pred']
    ssr = np.sum((y_true - y_pred) ** 2)
    tss = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ssr / tss)

# Initialize a dictionary to store combined dataframes and R² values for each factor
pls_combined_dataframes = {}
pls_r2_results = {}

# Load the dataframes for each factor, calculate R², and store them in the dictionary
factors = [1, 2, 3, 4, 5, 6]  # List of factors
for n_factors in tqdm(factors, desc='PLS'):
    combined_df = load_and_process_predictions(hdf5_path, n_factors)
    if combined_df is not None:
        pls_combined_dataframes[n_factors] = combined_df
        pls_r2_results[f'PLS_factor_{n_factors}'] = calculate_r2(combined_df)

# Create DataFrame for the R² results
r2_pls_df = pd.DataFrame.from_dict(pls_r2_results, orient='index', columns=['R^2'])

# Pivot the DataFrame for better visualization
r2_pls_df_pivoted = r2_pls_df.T

# Function to convert DataFrame to LaTeX code
def dataframe_to_latex(df):
    return df.to_latex(index=False)

# Generate LaTeX code for the pivoted DataFrame
r2_pls_latex = dataframe_to_latex(r2_pls_df_pivoted)

# Display the results
print("R^2 PLS DataFrame:")
print(r2_pls_df_pivoted)

print("\nR^2 PLS LaTeX:")
print(r2_pls_latex)


#**Calculation of $R^2$ for the FF models**

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import r2_score

# Function to load and concatenate predictions
def load_concatenated_predictions(hdf5_path, factor_sequences):
    concatenated_predictions = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        all_keys = store.keys()
        for n_factors in range(1, len(factor_sequences) + 1):
            factor_dfs = []
            for fold_key in all_keys:
                if f'/predictions/n_factors_{n_factors}_fold_' in fold_key:
                    factor_df = store[fold_key]
                    factor_dfs.append(factor_df)
            if factor_dfs:
                concatenated_predictions[n_factors] = pd.concat(factor_dfs, ignore_index=True)
    return concatenated_predictions

# Function to calculate R^2 for each factor sequence
def calculate_r2(concatenated_predictions):
    r2_scores = {}
    for n_factors, df in concatenated_predictions.items():
        y_true = df['y_true']
        y_pred = df['y_pred']
        r2 = r2_score(y_true, y_pred)
        r2_scores[n_factors] = r2
    return r2_scores

# Paths to HDF5 files
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_FF')
fwd_hdf5_path = results_path / 'fwd_predictions.h5'
pred_hdf5_path = results_path / 'predictions.h5'

# Factors sequences to use
factor_sequences = [
    ['Mkt-RF'],
    ['Mkt-RF', 'SMB'],
    ['Mkt-RF', 'SMB', 'HML'],
    ['Mkt-RF', 'SMB', 'HML', 'Mom'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA', 'RMW'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA', 'RMW', 'Mom']
]

# Load concatenated predictions for both paths
fwd_concatenated_predictions = load_concatenated_predictions(fwd_hdf5_path, factor_sequences)
pred_concatenated_predictions = load_concatenated_predictions(pred_hdf5_path, factor_sequences)

# Calculate R^2 for each path
fwd_r2_scores = calculate_r2(fwd_concatenated_predictions)
pred_r2_scores = calculate_r2(pred_concatenated_predictions)

# Create DataFrame for LaTeX table
r2_df = pd.DataFrame({
    'factor_1': [fwd_r2_scores.get(1, float('nan')), pred_r2_scores.get(1, float('nan'))],
    'factor_2': [fwd_r2_scores.get(2, float('nan')), pred_r2_scores.get(2, float('nan'))],
    'factor_3': [fwd_r2_scores.get(3, float('nan')), pred_r2_scores.get(3, float('nan'))],
    'factor_4': [fwd_r2_scores.get(4, float('nan')), pred_r2_scores.get(4, float('nan'))],
    'factor_5': [fwd_r2_scores.get(5, float('nan')), pred_r2_scores.get(5, float('nan'))],
    'factor_6': [fwd_r2_scores.get(6, float('nan')), pred_r2_scores.get(6, float('nan'))]
}, index=['fwd_predictions', 'predictions'])

# Convert the DataFrame to LaTeX format
latex_table = r2_df.to_latex(float_format="%.5f")
print(latex_table)
